# Imports

In [1]:
import numpy as np
import scipy.io
import matplotlib.pyplot as plt
from scipy.signal import butter, lfilter
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# EEGNet-specific imports
from EEGModels import EEGNet
from tensorflow.keras import utils as np_utils
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras import backend as K

import tensorflow.keras as keras
from tensorflow.keras.models import Model
from tensorflow.keras import layers
from tensorflow.keras.regularizers import l1_l2
from tensorflow.keras.constraints import max_norm
from tensorflow.keras import backend as K
import tensorflow.keras.models as models
import tensorflow.compat.v1 as tf
import numpy as np
from tslearn.metrics import soft_dtw
import os
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Activation, Permute, Dropout, Input
from tensorflow.keras.layers import Conv2D, MaxPooling2D, AveragePooling2D
from tensorflow.keras.layers import SeparableConv2D, DepthwiseConv2D
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import SpatialDropout2D
from tensorflow.keras.regularizers import l1_l2
from tensorflow.keras.layers import Input, Flatten
from tensorflow.keras.constraints import max_norm
from tensorflow.keras import backend as K
from scipy.signal import butter, filtfilt
from sklearn.model_selection import KFold
from scipy import io, signal
import math

# Parameters

In [2]:
# Parameters of benchmark dataset
chans = 64
n_subjects = 5
n_trials = 6
fs = 250
n_freqs = 10
#gap = 4
t_cue = 0.5

# Parameters for EEGNet classification
kernels = 1
t_select = 1
t_start = 2
n_start = t_start * fs
t_sum = t_cue + t_select

# Parameters for Butterworth filter
lowcut1 = 7
lowcut2 = 14
lowcut3 = 21
highcut = 90

# Parameters for data segment
seg_length = int(t_select * fs)
n_seg = 4

# Experiment
leave_one_out = False 

# Functions

In [3]:
def get_segmented_data(data, num_segments, segment_length):
    num_targets, num_channels, _, num_trials = data.shape
    required_length = num_segments * segment_length

    if required_length == 0:
        return np.array([])  

    segmented_data = np.zeros((num_targets, num_channels, num_trials, num_segments, segment_length))

    for target in range(num_targets):
        for channel in range(num_channels):
            for trial in range(num_trials):
                trial_data = data[target, channel, :, trial]

                if len(trial_data) < required_length:
                    continue  

                for i in range(num_segments):
                    start_index = i * segment_length
                    end_index = start_index + segment_length
                    segmented_data[target, channel, trial, i, :] = trial_data[start_index:end_index]

    return segmented_data

In [4]:
def bandpass_filter(data, lowcut, highcut, fs, order=5):
    nyq  = 0.5 * fs  # Nyquist Frequency
    low  = lowcut / nyq
    high = highcut /nyq
    b, a = butter(order, [low, high], btype='band')
    y    = lfilter(b, a, data)
    return y

def filter_set(data_set, lowcut, highcut, fs, order=5):
    filtered_set = np.zeros_like(data_set)
    for i in range(data_set.shape[0]):
        for j in range(data_set.shape[1]):
            filtered_set[i, j, :] = bandpass_filter(data_set[i, j, :], lowcut, highcut, fs, order=5)
    return filtered_set

In [5]:
layers_i = 0

def build_fusion_model():
        global layers_i
        model1 = EEGNet(nb_classes = n_freqs, Chans = chans, Samples = samples, dropoutRate = 0.5, kernLength = 125, F1 = 8, D = 2, F2 = 16, 
               dropoutType = 'Dropout')

        for layer in model1.layers:
            layers_i = layers_i + 1
            layer.name = layer.name + str(layers_i)
    
        model2 = EEGNet(nb_classes = n_freqs, Chans = chans, Samples = samples, dropoutRate = 0.5, kernLength = 125, F1 = 8, D = 2, F2 = 16, 
               dropoutType = 'Dropout')

        for layer in model2.layers:
            layers_i = layers_i + 1
            layer.name = layer.name + str(layers_i)
            
        model3 = EEGNet(nb_classes = n_freqs, Chans = chans, Samples = samples, dropoutRate = 0.5, kernLength = 125, F1 = 8, D = 2, F2 = 16, 
               dropoutType = 'Dropout')

        for layer in model3.layers:
            layers_i = layers_i + 1
            layer.name = layer.name + str(layers_i)
            
        features = layers.concatenate([model1.layers[-2].output, model2.layers[-2].output, model3.layers[-2].output])
        features = layers.Dense(10, activation="softmax")(features)
        fusion_model = models.Model([model1.input, model2.input, model3.input], features)
        fusion_model.compile(loss='categorical_crossentropy', optimizer = "Adam", metrics = ["accuracy"])
        return fusion_model

In [6]:
layers_i = 0

def build_fusion_model1():
        global layers_i
        model1 = EEGNet(nb_classes = n_freqs, Chans = chans, Samples = samples, dropoutRate = 0.5, kernLength = 125, F1 = 8, D = 2, F2 = 16, 
               dropoutType = 'Dropout')

        for layer in model1.layers:
            layers_i = layers_i + 1
            layer.name = layer.name + str(layers_i)
    
        model2 = EEGNet(nb_classes = n_freqs, Chans = chans, Samples = samples, dropoutRate = 0.5, kernLength = 125, F1 = 8, D = 2, F2 = 16, 
               dropoutType = 'Dropout')

        for layer in model2.layers:
            layers_i = layers_i + 1
            layer.name = layer.name + str(layers_i)
            
        model3 = EEGNet(nb_classes = n_freqs, Chans = chans, Samples = samples, dropoutRate = 0.5, kernLength = 125, F1 = 8, D = 2, F2 = 16, 
               dropoutType = 'Dropout')
            
        features = layers.concatenate([model1.layers[-2].output, model2.layers[-2].output])
        features = layers.Dense(10, activation="softmax")(features)
        fusion_model = models.Model([model1.input, model2.input], features)
        fusion_model.compile(loss='categorical_crossentropy', optimizer = "Adam", metrics = ["accuracy"])
        return fusion_model

In [7]:
def get_train_test(train_indices, test_indices, segmented_data):
    # training and test set, adapt the format to input to the EEGNet
    X_train   = segmented_data[:, :, train_indices, :, :] 
    targets1  = X_train.shape[0]
    channels1 = X_train.shape[1]
    trials1   = X_train.shape[2]
    segments1 = X_train.shape[3]
    samples1  = X_train.shape[4]
    X_train   = X_train.transpose(3, 0, 2, 1, 4).reshape(segments1*targets1*trials1, channels1, samples1)
        
    X_test    = segmented_data[:, :, test_indices, :, :]
    targets2  = X_test.shape[0]
    channels2 = X_test.shape[1]
    trials2   = X_test.shape[2]
    segments2 = X_test.shape[3]
    samples2  = X_test.shape[4]
    X_test    = X_test.transpose(3, 0, 2, 1, 4).reshape(segments2*targets2*trials2, channels2, samples2)
        
    # generaate labels for training and test set
    Y_train = []
    for i in range(segments1):
        for j in range(targets1):
            for k in range(trials1):
                Y_train.append(j)
    Y_train = np_utils.to_categorical(Y_train)
        
    Y_test = []
    for i in range(segments2):
        for j in range(targets2):
            for k in range(trials2):
                Y_test.append(j)
    Y_test = np_utils.to_categorical(Y_test)
    
    return X_train, X_test, Y_train, Y_test

In [8]:
def calculate_itr(classification_acc, num_targets, target_selection_time_seconds):
    p = classification_acc
    logp = np.log2(p)
    ip = 1.0 - p

    a = np.log2(num_targets)
    b = p * logp
    c = ip * np.log2( ip/(num_targets-1) )
    result = a + b + c
    return result * (60 / target_selection_time_seconds)

# Load subject data

In [9]:
selected_subjects_keys = [f"S{i}" for i in range(1, n_subjects+1)] 
seg_data = {}
data = {}
for i in range(1, n_subjects+1): 
    file_path = f'D://2024//7-9//dissertation//code//S{i}.mat'
    data[f'S{i}'] = scipy.io.loadmat(file_path)


for key in selected_subjects_keys:
    subject_data = data[key]['data'][:chans, n_start:, : n_freqs, :] # channels, samples, targets, trials
    subject_data = np.array(subject_data)
    subject_data = subject_data.transpose(2, 0, 1, 3) # targets, channels, samples, trials
    seg_data[key] = get_segmented_data(subject_data, n_seg, seg_length) # targets, channels, trials, segments, samples

data = {}

# Experiment - EEGNet

In [10]:
acc_eeg = []
itr_eeg = []
preds_eeg = []

for key in selected_subjects_keys:   
    for block in range(n_trials):
        train_i = []
        for k in range(n_trials):
            if k != block:
                train_i.append(k)
        test_i = [block]
        
        # Get train and test sets based on Leave One Out indicies 
        X_train, X_test, Y_train, Y_test = get_train_test(train_i, test_i, seg_data[key])
        
        # Filter data sets
        X1_train = filter_set(X_train, lowcut1, highcut, fs, order=5)
        X1_test = filter_set(X_test, lowcut1, highcut, fs, order=5)

        # Convert filtered data to NHWC (trials, channels, samples, kernels) format. Data
        samples  = seg_length
        
        X1_train = X1_train.reshape(X1_train.shape[0], chans, samples, kernels)
        X1_test  = X1_test.reshape(X1_test.shape[0], chans, samples, kernels)

        #Train EEGNet model
        eeg_model = model = EEGNet(nb_classes = n_freqs, Chans = chans, Samples = samples, 
               dropoutRate = 0.5, kernLength = 125, F1 = 8, D = 2, F2 = 16, 
               dropoutType = 'Dropout')
        
        eeg_model.compile(loss='categorical_crossentropy', optimizer='adam', 
              metrics = ['accuracy'])
        
        eeg_model.fit([X1_train], Y_train, batch_size=16, epochs=75, shuffle=True, verbose=0, 
        validation_data=([X1_test], Y_test))
        
        #Assess accuracy
        score = eeg_model.evaluate([X1_test], Y_test, verbose=0)
        #preds_eeg.append(eeg_model.predict(X_test))
        print(f"EEGNet acc: {score[1]}", flush=True)
        acc_eeg.append(score[1])

        #Assess ITR
        itr = calculate_itr(score[1], n_freqs, t_select)
        print(f"EEGNet itr: {itr}", flush=True)
        itr_eeg.append(itr)

print("EEGNet avg. acc: " + str(np.mean(acc_eeg))) # The average accuracy of all subjects
print("EEGNet avg. std: " + str(np.std(acc_eeg)))
print("EEGNet avg. ITR: " + str(np.mean(itr_eeg)))

C:\Users\qianqian\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


EEGNet acc: 0.75
EEGNet itr: 68.72674880270606
EEGNet acc: 0.5249999761581421
EEGNet itr: 32.7207104456012
EEGNet acc: 0.699999988079071
EEGNet itr: 59.58638571453889
EEGNet acc: 0.7250000238418579
EEGNet itr: 64.06602600468118
EEGNet acc: 0.625
EEGNet itr: 47.150888656861206
EEGNet acc: 0.699999988079071
EEGNet itr: 59.58638571453889
EEGNet avg. acc: 0.6708333293596903
EEGNet avg. std: 0.07557649709544052
EEGNet avg. ITR: 55.30619088982124


# Experiment - Fusion Model

In [10]:
acc_fusion = []
itr_fusion = []
preds_fusion = []

for key in selected_subjects_keys:   
    for block in range(n_freqs):
        train_i = []
        for k in range(0,1): #n_trials
            if k != block:
                train_i.append(k)
        test_i = [block]
        
        #Get train and test sets based on Leave One Out indicies 
        X_train, X_test, Y_train, Y_test = get_train_test(train_i, test_i, seg_data[key])
        
        # Filter data sets
        X1_train = filter_set(X_train, lowcut1, highcut, fs, order=5)
        X2_train = filter_set(X_train, lowcut2, highcut, fs, order=5)
        X3_train = filter_set(X_train, lowcut3, highcut, fs, order=5)
        
        X1_test = filter_set(X_test, lowcut1, highcut, fs, order=5)
        X2_test = filter_set(X_test, lowcut2, highcut, fs, order=5)
        X3_test = filter_set(X_test, lowcut3, highcut, fs, order=5)

        # Convert filtered data to NHWC (trials, channels, samples, kernels) format. Data
        samples  = seg_length
        
        X1_train = X1_train.reshape(X1_train.shape[0], chans, samples, kernels)
        X2_train = X2_train.reshape(X2_train.shape[0], chans, samples, kernels)
        X3_train = X3_train.reshape(X3_train.shape[0], chans, samples, kernels)

        X1_test  = X1_test.reshape(X1_test.shape[0], chans, samples, kernels)
        X2_test  = X2_test.reshape(X2_test.shape[0], chans, samples, kernels)
        X3_test  = X3_test.reshape(X3_test.shape[0], chans, samples, kernels)

        #Train fusion model
        fusion_model = build_fusion_model()
        fusion_model.fit([X1_train, X2_train, X3_train], Y_train, batch_size=16, epochs=75, shuffle=True, verbose=0, 
        validation_data=([X1_test, X2_test, X3_test], Y_test))
        
        #Assess accuracy
        score_FB = fusion_model.evaluate([X1_test, X2_test, X3_test], Y_test, verbose=0)
        #preds_eeg.append(eeg_model.predict(X_test))
        print(f"FB-EEGNet acc: {score_FB[1]}", flush=True)
        acc_fusion.append(score_FB[1])

        #Assess ITR
        itr_FB = calculate_itr(score_FB[1], n_freqs, t_select)
        print(f"FB-EEGNet itr: {itr_FB}", flush=True)
        itr_fusion.append(itr_FB)

print("FB-EEGNet avg. acc: " + str(np.mean(acc_fusion))) # The average accuracy of all subjects
print("FB-EEGNet avg. std: " + str(np.std(acc_fusion)))
print("FB-EEGNet avg. ITR: " + str(np.mean(itr_fusion)))

ValueError: zero-size array to reduction operation maximum which has no identity

# Experiment - Fusion model with 2 filter banks

In [26]:
acc_fusion = []
itr_fusion = []
preds_fusion = []

for key in selected_subjects_keys:   
    for block in range(n_trials):
        train_i = []
        for k in range(n_trials):
            if k != block:
                train_i.append(k)
        test_i = [block]
        
        #Get train and test sets based on Leave One Out indicies 
        X_train, X_test, Y_train, Y_test = get_train_test(train_i, test_i, seg_data[key])
        
        # Filter data sets
        X1_train = filter_set(X_train, lowcut1, highcut, fs, order=5)
        X2_train = filter_set(X_train, lowcut2, highcut, fs, order=5)
        
        X1_test = filter_set(X_test, lowcut1, highcut, fs, order=5)
        X2_test = filter_set(X_test, lowcut2, highcut, fs, order=5)

        # Convert filtered data to NHWC (trials, channels, samples, kernels) format. Data
        samples  = seg_length
        
        X1_train = X1_train.reshape(X1_train.shape[0], chans, samples, kernels)
        X2_train = X2_train.reshape(X2_train.shape[0], chans, samples, kernels)

        X1_test  = X1_test.reshape(X1_test.shape[0], chans, samples, kernels)
        X2_test  = X2_test.reshape(X2_test.shape[0], chans, samples, kernels)

        #Train fusion model
        fusion_model = build_fusion_model1()
        fusion_model.fit([X1_train, X2_train], Y_train, batch_size=16, epochs=75, shuffle=True, verbose=0, 
        validation_data=([X1_test, X2_test,], Y_test))
        
        #Assess accuracy
        score_FB = fusion_model.evaluate([X1_test, X2_test,], Y_test, verbose=0)
        #preds_eeg.append(eeg_model.predict(X_test))
        print(f"FB-EEGNet acc: {score_FB[1]}", flush=True)
        acc_fusion.append(score_FB[1])

        #Assess ITR
        itr_FB = calculate_itr(score_FB[1], n_freqs, t_select)
        print(f"FB-EEGNet itr: {itr_FB}", flush=True)
        itr_fusion.append(itr_FB)

print("FB-EEGNet avg. acc: " + str(np.mean(acc_fusion))) # The average accuracy of all subjects
print("FB-EEGNet avg. std: " + str(np.std(acc_fusion)))
print("FB-EEGNet avg. ITR: " + str(np.mean(itr_fusion)))

FB-EEGNet acc: 0.800000011920929
FB-EEGNet itr: 117.9609036805108
FB-EEGNet avg. acc: 0.800000011920929
FB-EEGNet avg. std: 0.0
FB-EEGNet avg. ITR: 117.9609036805108
